# Incendios de magnitud chile
### Objetivos: Relacionar la meteorologia con los incendios de magnitud  en chile. identificar patrones...

In [22]:
# Si estás en un notebook y geopandas no está instalado, descomenta e instala:
%pip install geopandas
%pip install matplotlib

import geopandas as gpd
import matplotlib.pyplot as plt

# Usa raw string para evitar problemas con backslashes en Windows
shp_path = r"C:\Users\luisc\Documents\Github\analisis-geoespacial\incendios-magnitud-meteorologia\incendios\poligonos\if_magnitud_2013_2014\if_magnitud_2013_2014.shp"


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Auditoría y compilación de capas

Las capas no tienen exactamente el mismo esquema: las primeras temporadas usan nombres en mayúsculas, desde 2021 aparecen variaciones y 2023-2024 contiene campos adicionales. Se normalizan los campos comunes, se conservan los campos exclusivos y se añade `CAPA_ORIGEN`. Todas las geometrías se llevan a `EPSG:32719`; la capa de Isla de Pascua parte en `EPSG:32712`.

## Credenciales meteorológicas

La descarga de `reanalysis-era5-land` usa **Copernicus Climate Data Store (CDS)** y `cdsapi`, no la API de ECMWF (`api.ecmwf.int/v1`). Se requiere una credencial CDS configurada en `%USERPROFILE%\\.cdsapirc` o mediante `CDSAPI_KEY`. El archivo `API.INFO.txt` de ECMWF no se utiliza automáticamente y no sirve para esta solicitud.

In [23]:
from pathlib import Path
import pandas as pd

root = Path(r"C:\Users\luisc\Documents\Github\analisis-geoespacial\incendios-magnitud-meteorologia\incendios\poligonos")
output_path = root / "if_magnitud_compilado.gpkg"
target_crs = "EPSG:32719"

# Alias de los campos que representan la misma variable en distintas temporadas.
aliases = {
    "ID": "ID",
    "id": "ID",
    "TEMPORADA": "TEMPORADA",
    "temporada": "TEMPORADA",
    "NOM_INCEN": "NOM_INCEN",
    "nom_incen": "NOM_INCEN",
    "CAUSA": "CAUSA",
    "causa": "CAUSA",
    "SUPERFICIE": "SUPERFICIE",
    "superficie": "SUPERFICIE",
    "CODREG": "CODREG",
    "codreg": "CODREG",
    "CODPROV": "CODPROV",
    "codprov": "CODPROV",
    "CODCOM": "CODCOM",
    "codcom": "CODCOM",
    "REGION": "REGION",
    "region": "REGION",
    "COMUNA": "COMUNA",
    "comuna": "COMUNA",
    "PROVINCIA": "PROVINCIA",
    "provincia": "PROVINCIA",
    "FECHA_INI": "FECHA_INI",
    "fecha_ini": "FECHA_INI",
    "FH_INICIO": "FECHA_INI",
    "FECHA_TER": "FECHA_TER",
    "fecha_ter": "FECHA_TER",
    "FH_EXTINC": "FECHA_TER",
}

capas = []
for shp_path in sorted(root.glob("*/if_magnitud_*.shp")):
    capa = gpd.read_file(shp_path)
    capa = capa.to_crs(target_crs)
    capa = capa.rename(columns={
        nombre: aliases.get(nombre, nombre.upper())
        for nombre in capa.columns
        if nombre != capa.geometry.name
    })
    capa["CAPA_ORIGEN"] = shp_path.parent.name
    capa["TEMPORADA_ORIGEN"] = shp_path.parent.name.replace("if_magnitud_", "")
    capas.append(capa)

compilado = gpd.GeoDataFrame(pd.concat(capas, ignore_index=True, sort=False), crs=target_crs)

if output_path.exists():
    output_path.unlink()
compilado.to_file(output_path, layer="incendios_magnitud", driver="GPKG")

print(f"Capas compiladas: {len(capas)}")
print(f"Registros compilados: {len(compilado)}")
print(f"Campos finales: {len(compilado.columns)}")
print(f"CRS final: {compilado.crs}")
print(f"Geometrías inválidas: {(~compilado.geometry.is_valid).sum()}")
display(compilado.groupby("CAPA_ORIGEN").size().rename("registros").to_frame())
display(compilado.head())

Capas compiladas: 13
Registros compilados: 781
Campos finales: 24
CRS final: EPSG:32719
Geometrías inválidas: 77


,registros
CAPA_ORIGEN,
if_magnitud_2013_2014,51
if_magnitud_2014_2015,79
if_magnitud_2015_2016,33
if_magnitud_2016_2017,144
if_magnitud_2017_2018,21
if_magnitud_2018_2019,67
if_magnitud_2019_2020,62
if_magnitud_2020_2021,20
if_magnitud_2021_2022,62


,ID,TEMPORADA,NOM_INCEN,CAUSA,SUPERFICIE,CODREG,CODPROV,CODCOM,REGION,COMUNA,...,SHAPE_AREA,geometry,CAPA_ORIGEN,TEMPORADA_ORIGEN,NUMERO_REG,AMBITO,UTM_E,UTM_N,HUSO,SUP
0,1.0,2013-2014,Santa Zenada-deuco,Otros incendios no clasificados,574.0,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,5.739890e+06,"POLYGON ((1.73e+05 5.81e+06, 1.73e+05 5.81e+06...",if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,2013-2014,Fundo Monaco,Incendio Intencional,207.7,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,2.076936e+06,"POLYGON ((1.77e+05 5.81e+06, 1.77e+05 5.81e+06...",if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,2013-2014,Toquihue,Incendio Intencional,891.3,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,8.912981e+06,"POLYGON ((1.96e+05 5.77e+06, 1.96e+05 5.77e+06...",if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,NaN,NaN,NaN
3,28.0,2013-2014,Rumena,Incendios intencionales,3268.9,08,None,None,RegiÃ³n del BiobÃ­o,None,...,3.268943e+07,"POLYGON ((9.04e+04 5.87e+06, 9.04e+04 5.87e+06...",if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,NaN,NaN,NaN
4,4.0,2013-2014,Miraflores,ElaboraciÃ³n de carbÃ³n,1057.8,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,1.057828e+07,"POLYGON ((1.67e+05 5.8e+06, 1.67e+05 5.8e+06, ...",if_magnitud_2013_2014,2013_2014,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Tabla de atributos del archivo compilado
from pathlib import Path
import geopandas as gpd

archivo_compilado = Path(
    r"C:\Users\luisc\Documents\Github\analisis-geoespacial\incendios-magnitud-meteorologia\incendios\poligonos\if_magnitud_compilado.gpkg"
)
tabla_atributos = gpd.read_file(archivo_compilado, layer="incendios_magnitud")

print(f"Registros: {len(tabla_atributos)}")
print(f"Campos: {len(tabla_atributos.columns)}")
print(f"CRS: {tabla_atributos.crs}")

display(tabla_atributos)

Registros: 781
Campos: 24
CRS: EPSG:32719


,ID,TEMPORADA,NOM_INCEN,CAUSA,SUPERFICIE,CODREG,CODPROV,CODCOM,REGION,COMUNA,...,SHAPE_AREA,CAPA_ORIGEN,TEMPORADA_ORIGEN,NUMERO_REG,AMBITO,UTM_E,UTM_N,HUSO,SUP,geometry
0,1.0,2013-2014,Santa Zenada-deuco,Otros incendios no clasificados,574.000000,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,5.739890e+06,if_magnitud_2013_2014,2013_2014,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((1.73e+05 5.81e+06, 1.73e+05 5...."
1,2.0,2013-2014,Fundo Monaco,Incendio Intencional,207.700000,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,2.076936e+06,if_magnitud_2013_2014,2013_2014,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((1.77e+05 5.81e+06, 1.77e+05 5...."
2,3.0,2013-2014,Toquihue,Incendio Intencional,891.300000,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,8.912981e+06,if_magnitud_2013_2014,2013_2014,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((1.96e+05 5.77e+06, 1.96e+05 5...."
3,28.0,2013-2014,Rumena,Incendios intencionales,3268.900000,08,None,None,RegiÃ³n del BiobÃ­o,None,...,3.268943e+07,if_magnitud_2013_2014,2013_2014,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((9.04e+04 5.87e+06, 9.04e+04 5...."
4,4.0,2013-2014,Miraflores,ElaboraciÃ³n de carbÃ³n,1057.800000,09,None,None,RegiÃ³n de La AraucanÃ­a,None,...,1.057828e+07,if_magnitud_2013_2014,2013_2014,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((1.67e+05 5.8e+06, 1.67e+05 5.8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
776,NaN,2024-2025,83 - FUNDO EL ESCORIAL,1.2.2. PartÃ­culas incandescentes generadas p...,478.022342,6,62,6204,O'Higgins,Marchigue,...,NaN,if_magnitud_2024_2025,2024_2025,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((2.51e+05 6.19e+06, 2.51e+05 6...."
777,NaN,2024-2025,274 - VEGA HONDA,4.2.1. Quema no avisada de desechos agrÃ­colas...,774.982079,16,161,16108,Ãuble,San Ignacio,...,NaN,if_magnitud_2024_2025,2024_2025,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((2.35e+05 5.93e+06, 2.35e+05 5...."
778,NaN,2024-2025,290 - PATAGUAL,4.2.1. Quema no avisada de desechos agrÃ­colas...,219.833516,16,161,16106,Ãuble,Pinto,...,NaN,if_magnitud_2024_2025,2024_2025,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((2.37e+05 5.94e+06, 2.37e+05 5...."
779,NaN,2024-2025,306 - SAN PATRICIO,4.11.3. Empleo de fuentes de calor en faena de...,1938.760474,16,163,16302,Ãuble,Coihueco,...,NaN,if_magnitud_2024_2025,2024_2025,NaN,None,None,None,None,NaN,"MULTIPOLYGON (((2.68e+05 5.92e+06, 2.68e+05 5...."


# Extraccion datos meteorologicos


In [25]:
%pip install -q cdsapi xarray netcdf4

Note: you may need to restart the kernel to use updated packages.


In [26]:
import cdsapi
import numpy as np
import xarray as xr
from pathlib import Path
import os

# Descarga ERA5-Land para el primer incendio y calcula viento y humedad relativa.
# Requiere credenciales CDS configuradas mediante CDSAPI_KEY o ~/.cdsapirc.


# Primer incendio
incendio = tabla_atributos.iloc[0]

fecha_inicio = pd.to_datetime(incendio["FECHA_INI"], errors="coerce")
fecha_termino = pd.to_datetime(incendio["FECHA_TER"], errors="coerce")

if pd.isna(fecha_inicio):
    raise ValueError("El primer incendio no tiene una fecha de inicio válida.")

if pd.isna(fecha_termino):
    fecha_termino = fecha_inicio

# Coordenadas del centroide en WGS84
centroide = incendio.geometry.centroid
punto_wgs84 = (
    gpd.GeoSeries([centroide], crs=tabla_atributos.crs)
    .to_crs("EPSG:4326")
    .iloc[0]
)

latitud = float(punto_wgs84.y)
longitud = float(punto_wgs84.x)

# ERA5-Land trabaja con fechas y horas UTC
fechas = pd.date_range(fecha_inicio.normalize(), fecha_termino.normalize(), freq="D")
horas = [f"{hora:02d}:00" for hora in range(24)]

archivo_era5 = root / "era5_land_primer_incendio.nc"

solicitud = {
    "variable": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
    ],
    "year": sorted(fechas.strftime("%Y").unique().tolist()),
    "month": sorted(fechas.strftime("%m").unique().tolist()),
    "day": sorted(fechas.strftime("%d").unique().tolist()),
    "time": horas,
    "data_format": "netcdf",
    "download_format": "unarchived",
}

# No escribas la clave en el notebook ni la muestres en pantalla.
api_key = os.getenv("CDSAPI_KEY")
api_url = os.getenv("CDSAPI_URL", "https://cds.climate.copernicus.eu/api")
archivo_configuracion = Path.home() / ".cdsapirc"

if api_key:
    cliente_cds = cdsapi.Client(url=api_url, key=api_key)
elif archivo_configuracion.exists():
    cliente_cds = cdsapi.Client()
else:
    raise RuntimeError(
        "No hay credenciales CDS disponibles. Crea %USERPROFILE%\\.cdsapirc "
        "o configura CDSAPI_KEY en el entorno del kernel. La clave de "
        "api.ecmwf.int no es válida para cdsapi/CDS."
    )

cliente_cds.retrieve(
    "reanalysis-era5-land",
    solicitud,
    str(archivo_era5),
)

print(f"Incendio: {incendio.get('NOM_INCEN', 'sin nombre')}")
print(f"Periodo solicitado: {fecha_inicio} a {fecha_termino}")
print(f"Coordenadas: {latitud:.5f}, {longitud:.5f}")
print(f"Archivo descargado: {archivo_era5}")

# Extracción del píxel más cercano
ds = xr.open_dataset(archivo_era5)
punto = ds.sel(latitude=latitud, longitude=longitud, method="nearest")

# Conversión a unidades útiles
temperatura_c = punto["t2m"] - 273.15
punto_de_rocio_c = punto["d2m"] - 273.15
u = punto["u10"]
v = punto["v10"]

velocidad_viento = np.sqrt(u**2 + v**2)

# Dirección meteorológica: dirección desde la que sopla el viento
direccion_viento = (270 - np.degrees(np.arctan2(v, u))) % 360

# Humedad relativa aproximada mediante fórmula de Magnus
es = 6.112 * np.exp((17.67 * temperatura_c) / (temperatura_c + 243.5))
e = 6.112 * np.exp((17.67 * punto_de_rocio_c) / (punto_de_rocio_c + 243.5))
humedad_relativa = (100 * e / es).clip(0, 100)

datos_meteorologicos = xr.Dataset({
    "temperatura_2m_c": temperatura_c,
    "temperatura_rocio_2m_c": punto_de_rocio_c,
    "velocidad_viento_10m_ms": velocidad_viento,
    "direccion_viento_10m_grados": direccion_viento,
    "humedad_relativa_2m_pct": humedad_relativa,
}).to_dataframe().reset_index()

display(datos_meteorologicos.head())

RuntimeError: No hay credenciales CDS disponibles. Crea %USERPROFILE%\.cdsapirc o configura CDSAPI_KEY en el entorno del kernel. La clave de api.ecmwf.int no es válida para cdsapi/CDS.

## Alternativa sin credenciales: Open-Meteo

Si la configuración de CDS continúa fallando, esta alternativa obtiene datos históricos horarios de ERA5 mediante Open-Meteo y no requiere API key para uso no comercial.

In [27]:
import numpy as np
import pandas as pd
import requests

# Selecciona el incendio que quieres analizar.
incendio = tabla_atributos.iloc[0]
fecha_inicio = pd.to_datetime(incendio["FECHA_INI"], errors="coerce")
fecha_termino = pd.to_datetime(incendio["FECHA_TER"], errors="coerce")

if pd.isna(fecha_inicio):
    raise ValueError("El incendio no tiene una fecha de inicio válida.")
if pd.isna(fecha_termino):
    fecha_termino = fecha_inicio

# Open-Meteo requiere coordenadas geográficas WGS84.
centroide = incendio.geometry.centroid
punto_wgs84 = gpd.GeoSeries([centroide], crs=tabla_atributos.crs).to_crs("EPSG:4326").iloc[0]

parametros = {
    "latitude": float(punto_wgs84.y),
    "longitude": float(punto_wgs84.x),
    "start_date": fecha_inicio.strftime("%Y-%m-%d"),
    "end_date": fecha_termino.strftime("%Y-%m-%d"),
    "hourly": "temperature_2m,dew_point_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m",
    "wind_speed_unit": "ms",
    "timezone": "UTC",
}

respuesta = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params=parametros,
    timeout=60,
)
respuesta.raise_for_status()
resultado = respuesta.json()

if "hourly" not in resultado:
    raise RuntimeError(f"Open-Meteo no devolvió datos: {resultado}")

datos_meteorologicos = pd.DataFrame(resultado["hourly"])
datos_meteorologicos["time"] = pd.to_datetime(datos_meteorologicos["time"])

print(f"Incendio: {incendio.get('NOM_INCEN', 'sin nombre')}")
print(f"Periodo: {fecha_inicio.date()} a {fecha_termino.date()}")
print(f"Coordenadas WGS84: {punto_wgs84.y:.5f}, {punto_wgs84.x:.5f}")
print(f"Registros meteorológicos: {len(datos_meteorologicos)}")
display(datos_meteorologicos.head())

Incendio: Santa Zenada-deuco
Periodo: 2014-01-05 a 2014-01-05
Coordenadas WGS84: -37.82899, -72.71448
Registros meteorológicos: 24


,time,temperature_2m,dew_point_2m,wind_speed_10m,wind_direction_10m,relative_humidity_2m
0,2014-01-05 00:00:00,20.2,12.4,5.65,193,61
1,2014-01-05 01:00:00,18.5,12.3,5.58,195,67
2,2014-01-05 02:00:00,17.3,12.2,5.68,194,72
3,2014-01-05 03:00:00,16.3,11.9,5.80,195,75
4,2014-01-05 04:00:00,15.5,11.4,5.95,197,76
